# 01. Векторы и dev
T4. Код и данные загружаются из GitHub; результаты сохраняются на Google Drive. Укажи свою ветку в GIT_REF.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/danilov-e/lora-activation-steering.git"
GIT_REF = "main"  # Ветка или тег, куда загружен этот проект.
PROJECT = Path("/content/lora-activation-steering")

if not PROJECT.exists():
    subprocess.run(
        ["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT)], check=True
    )
print("Commit:", subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "--short", "HEAD"], text=True
).strip())
%pip -q install -r {PROJECT}/requirements.txt


## 1. Настройки
Укажи признак и имя запуска. Настройки фиксируются при первом выполнении; для изменений используй новое RUN_NAME. test_repeats задаёт число повторов в ноутбуке 03.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.dont_write_bytecode = True
sys.path.insert(0, str(PROJECT))
from src import workflow as w
from src import judge as j
import pandas as pd
from IPython.display import display, Image

RUNS = Path("/content/drive/MyDrive/lora_activation_steering/runs")
RUN_NAME = "research_01"
TRAIT = "sycophancy"  # sycophancy / evil / hallucination

CONFIG = w.default_config()
CONFIG["layers"] = [7, 14, 15, 21]
CONFIG["alphas"] = [-1.0, 0.5, 1.0, 2.0]
CONFIG["dev_repeats"] = 1
CONFIG["test_repeats"] = 3
RUN = w.create_run(PROJECT, RUN_NAME, TRAIT, CONFIG, runs_dir=RUNS)


## 2. Исходный вектор
100 готовых pos/neg-пар. Средние активации токенов ответа, вектор pos−neg без нормализации. Индексы слоёв начинаются с нуля.


In [ ]:
w.extract(RUN)
geometry, figure = w.extraction_report(RUN)
display(geometry.round(4))
display(Image(filename=str(figure)))


### Диагностика направления
12 dev-вопросов: изменение logP(pos)−logP(neg). Это проверка вероятностей готовых ответов; поведение оценивается ниже на свободной генерации.


In [ ]:
contrast, figure = w.contrast(RUN, n_questions=12)
display(contrast.round(4))
display(Image(filename=str(figure)))


## 3. Ответы dev
60 вопросов × (baseline + 16 комбинаций) = 1020 ответов. Каждая строка сохраняется сразу; повторный запуск продолжает генерацию.


In [ ]:
conditions = w.sweep_conditions(CONFIG)
print("Ответов:", len(w.pairs(RUN, "dev")) * len(conditions) * CONFIG["dev_repeats"])
DEV = w.generate(RUN, "dev", RUN/"dev", conditions, repeats=CONFIG["dev_repeats"])


## 4. OpenAI API
Ключ OpenAI: Colab Secrets → OPENAI_API_KEY или скрытый ввод. Платные запросы запускаются следующей ячейкой: пять ответов за запрос, только целевой признак и coherence, целые 0–100. Для стандартного dev — 204 запроса; сетевые повторы могут увеличить число. Сохранённые пакеты повторно не оцениваются.


In [ ]:
import os
from getpass import getpass
from google.colab import userdata

if not os.environ.get("OPENAI_API_KEY"):
    try:
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        key = None
    os.environ["OPENAI_API_KEY"] = key or getpass("OpenAI API key: ")

JUDGE_MODEL = "gpt-5.4-mini"
REASONING = "none"
display(j.judge(RUN/"dev", TRAIT, model=JUDGE_MODEL, reasoning=REASONING, batch_size=5))


## 5. Результаты
Средние, парные изменения от baseline и 95% bootstrap-интервалы по вопросам. На графиках — выраженность признака и связность; отдельно учитываются обрезанные ответы.


In [ ]:
summary, differences, figures = w.report(RUN/"dev", TRAIT)
display(summary.sort_values("target_mean", ascending=False).round(2))
display(differences.round(2))
for figure in figures:
    display(Image(filename=str(figure)))


In [ ]:
condition = summary[summary.condition.ne("base:baseline")].sort_values(
    "target_mean", ascending=False
).iloc[0].condition
print("Примеры для:", condition)
for _, row in w.examples(RUN/"dev", TRAIT, condition).iterrows():
    print("\n", row.question_id, row.condition, row[TRAIT], row.coherence)
    print(row.question)
    print(row.answer)


## 6. Выбор 2–3 настроек
Выбери по dev сочетания с заметным изменением признака и приемлемой coherence. Прочитай примеры. Заполни CHOICES: c1 — сильный эффект, c2 — лучше сохранена связность. Если эффект слабый, пересмотри сетку в новом запуске до LoRA. Test для выбора не используется.


In [ ]:
CHOICES = [
    # {"name": "c1", "layer": 14, "alpha": 1.0},
    # {"name": "c2", "layer": 15, "alpha": 0.5},
]
if CHOICES:
    display(w.save_choices(RUN, CHOICES))
else:
    print("Укажи 2–3 настройки по результатам dev. Примеры выше не являются рекомендацией.")
